In [1]:
import os, json, time, hmac, hashlib, requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field

load_dotenv()
BASE_URL = "https://jsonplaceholder.typicode.com"

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [3]:
# HTTP : 클라이언트 -> 요청을 보냅니다 (request)
#        서버 -> 응답 (response)

In [4]:
# request : 랭체인 설명해줘
    
# Method : 뭘 할건지?  POST/GET/PUT/DELETE
# URL : 어디서?        www.naver.com , BASE_URL
# 헤더 : 니가 누군데?  {인증정보, chrome/mozila ...}
# 바디 : 뭘 보낼건지?  {'title' : '제목입니다', 'content' : '내용입니다'}

In [5]:
# ok
# 200 : 조회 성공
# 201 : created 성공

# fail
# 400 : 요청 형식 틀림
# 401 : unauthorized
# 403 : forbidden

# 500 internal server error


In [6]:
r = requests.get(f"{BASE_URL}/posts/1")

In [7]:
r.status_code

200

In [8]:
r.json()

{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [9]:
r = requests.post(f"{BASE_URL}/posts", 
             json = {'title' : '새 글', 'body' : '본문', 'userId' : 1})

In [10]:
r.status_code

201

In [11]:
r.json()

{'title': '새 글', 'body': '본문', 'userId': 1, 'id': 101}

In [12]:
r = requests.delete(f"{BASE_URL}/posts/1", timeout=5)

In [13]:
r.status_code

200

In [14]:
r.json()

{}

In [15]:
session = requests.Session()
session.headers.update({
    'User-Agent' : 'abcde/1.0',
    'Accept' : 'application/json'
})

In [16]:
r = session.get(f"{BASE_URL}/posts", params = {"userId":1})

In [17]:
len(r.json())

10

In [18]:
r2 = session.get(f"{BASE_URL}/users/1")
r2.json()['name'], r2.json()['email']

('Leanne Graham', 'Sincere@april.biz')

In [19]:
r = session.post(f"{BASE_URL}/posts", json = {'title' : '새 글2', 'body' : '본문2', 'userId' : 1})

In [20]:
# curl -X POST "https://jsonplaceholder.typicode.com/posts" \
#     -H 

In [21]:
r.json()

{'title': '새 글2', 'body': '본문2', 'userId': 1, 'id': 101}

In [22]:
r.ok

True

In [24]:
# https://jsonplaceholder.typicode.com/posts

In [25]:
def rank_users_by_posts(user_ids):
    s = requests.Session()
    session.headers.update({
        'User-Agent' : 'abcde/1.0',
        'Accept' : 'application/json'
    })
    
    counts = []
    for uid in user_ids:
        r = session.get(f"{BASE_URL}/posts", params = {"userId": uid})
        counts.append({"userId" : uid, "post_count" : len(r.json())})
        
    counts.sort(key=lambda x : x["post_count"], reverse=True)
    for i, item in enumerate(counts, 1):
        item['rank'] = i
    
    return counts

In [26]:
rank_users_by_posts([1,2,3,4,5])

[{'userId': 1, 'post_count': 10, 'rank': 1},
 {'userId': 2, 'post_count': 10, 'rank': 2},
 {'userId': 3, 'post_count': 10, 'rank': 3},
 {'userId': 4, 'post_count': 10, 'rank': 4},
 {'userId': 5, 'post_count': 10, 'rank': 5}]

In [27]:
# https://jsonplaceholder.typicode.com/users/1
@tool
def fetch_user(user_id):
    """지정된 ID의 사용자 정보를 조회합니다"""
    r = requests.get(f"{BASE_URL}/users/{user_id}")
    if not r.ok:
        return f"error: status {r.status_code}"
    data = r.json()
    return json.dumps({
        'name' : data['name'],
        'email' : data['email'],
        'phone' : data['phone'],
        'company' : data['company']['name']
    }, ensure_ascii=False)

@tool
def fetch_user_post(user_id):
    """지정된 ID의 사용자가 작성한 게시글을 조회합니다"""
    r = requests.get(f"{BASE_URL}/posts", params={'userId':user_id})
    if not r.ok:
        return f"error: status {r.status_code}"
    posts = r.json()
    return json.dumps([{'id' : p['id'], 'title' : p['title']} for p in posts], ensure_ascii=False)

In [28]:
llm_api = llm.bind_tools([fetch_user, fetch_user_post])
response = llm_api.invoke("1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘")
print(f"호출된 도구 개수 : {len(response.tool_calls)}")

호출된 도구 개수 : 2


In [30]:
for tc in response.tool_calls:
    print(f"{tc['name']} : {tc['args']}")

fetch_user : {'user_id': 1}
fetch_user_post : {'user_id': 1}


In [32]:
tool_map = {'fetch_user' : fetch_user, 'fetch_user_post' : fetch_user_post}
messages = [HumanMessage(content = '1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘')]
resp = llm_api.invoke(messages)
messages.append(resp)

for tc in resp.tool_calls:
    result = tool_map[tc['name']].invoke(tc['args'])
    messages.append(ToolMessage(content=result, tool_call_id = tc['id']))
    
final = llm_api.invoke(messages)

In [33]:
final

AIMessage(content='1번 유저의 정보는 다음과 같습니다:\n\n- 이름: Leanne Graham\n- 이메일: Sincere@april.biz\n- 전화번호: 1-770-736-8031 x56442\n- 회사: Romaguera-Crona\n\n이 유저가 쓴 게시글의 갯수는 **10개**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 383, 'total_tokens': 458, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DXnsGYmYiJgc0IBF2kr3DjRWNHHOP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dba68-9ffa-7850-a84f-4706bb8cee13-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 383, 'output_tokens': 75, 'total_tokens': 458, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'a

In [34]:
print(final.content)

1번 유저의 정보는 다음과 같습니다:

- 이름: Leanne Graham
- 이메일: Sincere@april.biz
- 전화번호: 1-770-736-8031 x56442
- 회사: Romaguera-Crona

이 유저가 쓴 게시글의 갯수는 **10개**입니다.


In [ ]:
https://jsonplaceholder.typicode.com/todos/?userId=1

In [39]:
@tool
def fetch_todos(user_id) : 
    """지정된 사용자의 todo 목록 상위 5개를 조회합니다"""
    r = requests.get(f"{BASE_URL}/todos", params={'userID' : user_id})
    if not r.ok:
        return f"error : {r.status_code}"
    todos = r.json()[:5]
    return json.dumps(todos, ensure_ascii=False)

@tool
def count_complted_todos(user_id):
    """지정된 사용자의 완료된 todo 개수와 전체 개수를 계산합니다"""
    r = requests.get(f"{BASE_URL}/todos", params={'userID' : user_id})
    if not r.ok:
        return f"error : {r.status_code}"
    
    todos = r.json()
    completed = sum(1 for t in todos if t['completed'])
    total = len(todos)
    return json.dumps({
        'user_id' : user_id,
        'completed' : completed,
        'total' : total,
        'completion_rate' : round(completed/total * 100, 1) if total else 0,
        }, ensure_ascii=False)

In [40]:
todo_tools = [fetch_todos, count_complted_todos]
llm_todo = llm.bind_tools(todo_tools)
tool_map = {t.name:t for t in todo_tools}

messages = [HumanMessage(content = '1번 유저의 todo 완료율을 알려줘')]
resp = llm_todo.invoke(messages)
messages.append(resp)

print(f"tools : {[tc['name'] for tc in resp.tool_calls]}")

for tc in resp.tool_calls:
    result = tool_map[tc['name']].invoke(tc['args'])
    messages.append(ToolMessage(content=result, tool_call_id = tc['id']))
    
final = llm_todo.invoke(messages)

tools : ['count_complted_todos', 'fetch_todos']


In [42]:
print(final.content)

1번 유저의 todo 완료율은 45%입니다. 

총 200개의 todo 중 90개가 완료되었습니다.


In [ ]:
https://jsonplaceholder.typicode.com/posts

In [ ]:
https://jsonplaceholder.typicode.com/posts?_page=1&_limit=10